# 📊 DataCompass - Análise Exploratória de Dados

Este notebook permite fazer análises avançadas dos dados CSV recebidos via WhatsApp usando pandas, matplotlib e outras bibliotecas Python.

## 🎯 Funcionalidades
- Conexão direta com MongoDB
- Análises estatísticas avançadas com pandas
- Visualizações interativas
- Machine Learning exploratório
- Relatórios automatizados


In [4]:
# Instalação das dependências necessárias
%pip install pandas matplotlib seaborn plotly numpy scipy scikit-learn pymongo requests jupyter-lab


ERROR: Could not find a version that satisfies the requirement jupyter-lab (from versions: none)
ERROR: No matching distribution found for jupyter-lab
Note: you may need to restart the kernel to use updated packages.


In [5]:
# Imports necessários
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from pymongo import MongoClient
import requests
from datetime import datetime, timedelta
import json
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest

# Configurações
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Bibliotecas carregadas com sucesso!")

✅ Bibliotecas carregadas com sucesso!


## 🔌 Conexão com MongoDB e API


In [6]:
# Configurações de conexão
MONGODB_URI = "mongodb://localhost:27017/datacompass"
API_BASE_URL = "http://localhost:3000/api/whatsapp"

# Conectar ao MongoDB
try:
    client = MongoClient(MONGODB_URI)
    db = client.datacompass
    print(f"✅ Conectado ao MongoDB: {db.name}")

    # Testar conexão
    db.admin.command('ping')
    print("🏓 Ping MongoDB bem-sucedido")

    # Listar coleções disponíveis
    collections = db.list_collection_names()
    print(f"📂 Coleções disponíveis: {collections}")

except Exception as e:
    print(f"❌ Erro ao conectar ao MongoDB: {e}")
    client = None
    db = None

✅ Conectado ao MongoDB: datacompass
❌ Erro ao conectar ao MongoDB: 'Collection' object is not callable. If you meant to call the 'command' method on a 'Collection' object it is failing because no such method exists.


## 📱 Buscar Dados Recebidos via WhatsApp


In [8]:
def get_raw_data_from_api():
    """Busca todos os dados brutos da API"""
    try:
        response = requests.get(f"{API_BASE_URL}/raw")
        if response.status_code == 200:
            return response.json()
        else:
            print(f"❌ Erro na API: {response.status_code}")
            return None
    except Exception as e:
        print(f"❌ Erro ao acessar API: {e}")
        return None


def load_dataset_by_message_id(message_id):
    """Carrega um dataset específico pelo message_id"""
    try:
        response = requests.get(f"{API_BASE_URL}/raw/{message_id}")
        if response.status_code == 200:
            data = response.json()
            if data.get('success'):
                df = pd.DataFrame(data['data']['rawData'])
                return df, data['data']
            else:
                print(f"❌ Erro nos dados: {data.get('message')}")
                return None, None
        else:
            print(f"❌ Erro na API: {response.status_code}")
            return None, None
    except Exception as e:
        print(f"❌ Erro ao carregar dataset: {e}")
        return None, None


# Buscar dados disponíveis
print("🔍 Buscando dados disponíveis...")
raw_data_response = get_raw_data_from_api()

if raw_data_response and raw_data_response.get('success'):
    raw_datasets = raw_data_response['data']
    print(f"📊 Encontrados {len(raw_datasets)} datasets")

    for i, dataset in enumerate(raw_datasets):
        print(
            f"  {i+1}. {dataset['filename']} - {dataset['recordCount']} registros")
        print(f"     ID: {dataset['messageId'][:30]}...")
        print(
            f"     De: {dataset['from']} | Data: {dataset['timestamp'][:10]}")
        print()
else:
    print("❌ Nenhum dataset encontrado")
    raw_datasets = []

🔍 Buscando dados disponíveis...
📊 Encontrados 1 datasets
  1. vendas.csv - 20 registros
     ID: wamid.HBgMNTU4NDk4NjcxMTg4FQIA...


KeyError: 'timestamp'